# Stage 1 - EfficientNet-B3 on ODIR-5K + RFMiD (ver-5)

**Before running, click "Add Input" and attach both datasets:**
1. `Ocular Disease Recognition` (ODIR-5K) - same as ver-4
2. `Retinal Disease Classification` (RFMiD, by andrewmvd)

Settings: Accelerator = **GPU T4 x2 or P100**, Internet = **On** (for pretrained weights).

What changed vs ver-4:
- RFMiD train split is added to training, RFMiD validation to validation.
- RFMiD test split is kept as an **external test set** (never trained on).
- **Masked BCE**: labels a dataset does not provide (RFMiD has no cataract label) are ignored in the loss, not treated as 0.
- All images are cropped (black borders) before resize, same as the FastAPI service (`crop=True`).

Outputs in `/kaggle/working`: `b3_seed42.pth` + `ensemble_config.json` -> copy to `src/weights/`.

In [ ]:
!pip install timm albumentations grad-cam -q

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # must be set before CUDA is used

import re, json, random, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, f1_score, classification_report

warnings.filterwarnings("ignore")

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

seed_everything(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
INPUT_ROOT = "/kaggle/input"
print("Contents of /kaggle/input:", os.listdir(INPUT_ROOT))

# ---------------- ODIR-5K ----------------
xlsx_path, csv_path, img_dir = None, None, None
for r, dirs, files in os.walk(INPUT_ROOT):
    for f in files:
        if f.lower() == "data.xlsx" and xlsx_path is None:
            xlsx_path = os.path.join(r, f)
        if f.lower() == "full_df.csv" and csv_path is None:
            csv_path = os.path.join(r, f)
    for d in dirs:
        if d.lower() == "preprocessed_images" and img_dir is None:
            img_dir = os.path.join(r, d)

print("ODIR XLSX:", xlsx_path)
print("ODIR CSV :", csv_path)
print("ODIR IMGS:", img_dir)
if (xlsx_path is None and csv_path is None) or img_dir is None:
    raise FileNotFoundError("ODIR not found. Click 'Add Input' and add 'Ocular Disease Recognition'.")
print("ODIR images in folder:", len(os.listdir(img_dir)))

# ---------------- RFMiD ----------------
rfmid_csv = {}
for r, _, files in os.walk(INPUT_ROOT):
    for f in files:
        fl = f.lower()
        if fl.endswith('.csv') and 'rfmid' in fl:
            for split, key in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
                if key in fl and split not in rfmid_csv:
                    rfmid_csv[split] = os.path.join(r, f)
print("RFMiD CSVs:", rfmid_csv)
if len(rfmid_csv) < 3:
    raise FileNotFoundError("RFMiD not found. Click 'Add Input' and add 'Retinal Disease Classification' (andrewmvd).")

IMG_EXT = ('.png', '.jpg', '.jpeg', '.tif', '.tiff')

def image_index(csv_file):
    """Map image stem ('12') -> full path, searching next to the CSV (then one level up)."""
    for root in [os.path.dirname(csv_file), os.path.dirname(os.path.dirname(csv_file))]:
        idx = {}
        for r, _, files in os.walk(root):
            for f in files:
                if f.lower().endswith(IMG_EXT):
                    idx.setdefault(os.path.splitext(f)[0], os.path.join(r, f))
        if idx:
            return idx
    return {}

In [ ]:
df = pd.read_excel(xlsx_path) if xlsx_path is not None else pd.read_csv(csv_path)
print("ODIR patients in table:", len(df))

CLASSES = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
CLASS_NAMES = {
    'N': 'Normal', 'D': 'Diabetic Retinopathy', 'G': 'Glaucoma',
    'C': 'Cataract', 'A': 'AMD', 'H': 'Hypertensive Retinopathy',
    'M': 'Pathological Myopia', 'O': 'Other abnormalities'
}
NUM_CLASSES = len(CLASSES)
MASKS = [f'm_{c}' for c in CLASSES]   # 1 = label is known for this image, 0 = unknown (ignored in loss)

# ---------------- ODIR: labels from each eye's own keywords ----------------
def eye_labels(kw):
    kw = str(kw).lower()
    y = dict.fromkeys(CLASSES, 0)
    if 'normal fundus' in kw:
        y['N'] = 1
        return y
    if re.search(r'diabetic|proliferative', kw):  y['D'] = 1
    if 'glaucoma' in kw:                          y['G'] = 1
    if 'cataract' in kw:                          y['C'] = 1
    if re.search(r'age-related macular|amd', kw): y['A'] = 1
    if 'hypertensive' in kw:                      y['H'] = 1
    if 'myopia' in kw:                            y['M'] = 1
    if sum(y.values()) == 0:                      y['O'] = 1
    return y

rows = []
for _, r in df.iterrows():
    for side in ['Left', 'Right']:
        kw = r[f'{side}-Diagnostic Keywords']
        if re.search(r'no fundus image|anterior segment', str(kw).lower()):
            continue
        row = {'patient_id': f"odir_{r['ID']}", 'filename': r[f'{side}-Fundus'], 'keywords': kw}
        row.update(eye_labels(kw))
        rows.append(row)

eyes = pd.DataFrame(rows)
existing = set(os.listdir(img_dir))
eyes = eyes[eyes['filename'].isin(existing)].reset_index(drop=True)
eyes['path'] = eyes['filename'].apply(lambda x: os.path.join(img_dir, x))
eyes['source'] = 'ODIR'
for m in MASKS:
    eyes[m] = 1                       # ODIR gives all 8 labels
print("ODIR eye images:", len(eyes))

# ---------------- RFMiD: map 46 columns -> 8 classes ----------------
RFMID_MAP = {'D': ['DR'], 'G': ['ODC'], 'A': ['ARMD'], 'H': ['HR'], 'M': ['MYA']}

def load_rfmid(split):
    lab = pd.read_csv(rfmid_csv[split])
    idx = image_index(rfmid_csv[split])
    ids = lab['ID'].astype(str)
    disease_cols = [c for c in lab.columns if c not in ('ID', 'Disease_Risk')]
    mapped = {col for cols in RFMID_MAP.values() for col in cols}
    other_cols = [c for c in disease_cols if c not in mapped]

    out = pd.DataFrame({'patient_id': f"rfmid_{split}_" + ids,   # no patient IDs in RFMiD
                        'filename': ids, 'keywords': ''})
    out['path'] = ids.map(idx)
    healthy = (lab['Disease_Risk'] == 0)
    out['N'] = healthy.astype(int)
    for c, cols in RFMID_MAP.items():
        out[c] = (lab[[x for x in cols if x in lab.columns]].max(axis=1) > 0).astype(int)
    out['O'] = (lab[other_cols].sum(axis=1) > 0).astype(int)
    out['C'] = 0
    # diseased but no column set -> Other
    out.loc[~healthy & (out[['D', 'G', 'A', 'H', 'M', 'O']].sum(axis=1) == 0), 'O'] = 1

    for m in MASKS:
        out[m] = 1
    out['m_C'] = healthy.astype(int)   # RFMiD has no cataract label: known only for healthy eyes
    out['source'] = 'RFMiD'

    miss = out['path'].isna().sum()
    if miss:
        print(f"  RFMiD {split}: {miss} images not found -> dropped")
    return out[out['path'].notna()].reset_index(drop=True)

rfmid = {s: load_rfmid(s) for s in ['train', 'val', 'test']}
for s, d in rfmid.items():
    print(f"RFMiD {s}: {len(d)} images")
pd.DataFrame({s: d[CLASSES].sum() for s, d in rfmid.items()})

In [ ]:
counts = eyes[CLASSES].sum().sort_values(ascending=False)
print(counts)
print("\nImages with more than one disease:", (eyes[CLASSES].sum(axis=1) > 1).sum())

plt.figure(figsize=(8, 4))
counts.plot(kind='bar', color='steelblue')
plt.title("Class distribution (per eye)")
plt.ylabel("Number of images")
plt.xticks(rotation=0)
plt.show()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, c in zip(axes.ravel(), CLASSES):
    s = eyes[eyes[c] == 1].sample(1, random_state=1).iloc[0]
    img = cv2.cvtColor(cv2.imread(s['path']), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{c} - {CLASS_NAMES[c]}\n{s['filename']}")
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ODIR: patient-level split exactly like ver-4 (same seed -> same test patients)
patients = eyes.groupby('patient_id')[CLASSES].max().reset_index()
train_p, temp_p = train_test_split(patients, test_size=0.30, random_state=42, stratify=patients['D'])
val_p, test_p   = train_test_split(temp_p,  test_size=0.50, random_state=42, stratify=temp_p['D'])

odir_train = eyes[eyes['patient_id'].isin(train_p['patient_id'])]
odir_val   = eyes[eyes['patient_id'].isin(val_p['patient_id'])]
odir_test  = eyes[eyes['patient_id'].isin(test_p['patient_id'])]

# RFMiD: use its official splits. Test split = external test, never trained on.
train_df = pd.concat([odir_train, rfmid['train']], ignore_index=True)
val_df   = pd.concat([odir_val,   rfmid['val']],   ignore_index=True)
test_df  = odir_test.reset_index(drop=True)      # ODIR internal test (comparable to ver-4)
ext_df   = rfmid['test'].reset_index(drop=True)  # RFMiD external test

print(f"Train: {len(train_df)} | Val: {len(val_df)} | ODIR test: {len(test_df)} | RFMiD ext test: {len(ext_df)}")
print(pd.DataFrame({'train': train_df[CLASSES].sum(), 'val': val_df[CLASSES].sum(),
                    'odir_test': test_df[CLASSES].sum(), 'rfmid_test': ext_df[CLASSES].sum()}))
print("\nTrain by source:\n", train_df['source'].value_counts())

assert not set(train_df.patient_id) & set(val_df.patient_id)
assert not set(train_df.patient_id) & set(test_df.patient_id)
assert not set(val_df.patient_id) & set(test_df.patient_id)
print("\nNo patient leakage between splits ✔")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

IMG_SIZE = 384
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

def crop_black_borders(rgb, thr=10):
    """Crop the black background of a fundus photo and pad it to a square."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    ys, xs = np.where(gray > thr)
    if len(ys) < 100:
        return rgb
    rgb = rgb[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    h, w = rgb.shape[:2]
    s = max(h, w)
    canvas = np.zeros((s, s, 3), dtype=rgb.dtype)
    canvas[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = rgb
    return canvas

def enhance(rgb):
    """CLAHE on the lightness channel: boosts vessels/lesions and keeps the colors."""
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[..., 0] = clahe.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def prepare_rgb(rgb, crop=True):
    if crop:
        rgb = crop_black_borders(rgb)
    return cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)

def load_rgb(path, crop=True):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return prepare_rgb(img, crop)

# ---- Cache all images (crop + resize + CLAHE) in RAM once. RFMiD originals are big, so use threads ----
def build_cache(df):
    arr = np.zeros((len(df), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    def work(i_p):
        i, p = i_p
        arr[i] = enhance(load_rgb(p))
    with ThreadPoolExecutor(max_workers=4) as ex:
        list(tqdm(ex.map(work, enumerate(df['path'].values)), total=len(df), desc="caching"))
    return arr

train_arr = build_cache(train_df)
val_arr   = build_cache(val_df)
test_arr  = build_cache(test_df)
ext_arr   = build_cache(ext_df)
print("Cache size (GB):", round(sum(a.nbytes for a in [train_arr, val_arr, test_arr, ext_arr]) / 1e9, 2))

# ---- Augmentation (train only) ----
def make_compose(tfms):
    try:
        return A.Compose(tfms, seed=42)      # albumentations >= 2.0
    except TypeError:
        return A.Compose(tfms)               # older versions

train_tfms = make_compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-30, 30), p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.RandomGamma(p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
eval_tfms = A.Compose([A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

def to_tensor(rgb_resized):
    """Resized RGB -> normalized tensor [1,3,H,W] (same CLAHE + normalization as training)."""
    return eval_tfms(image=enhance(rgb_resized))['image'].unsqueeze(0)

class CachedDataset(Dataset):
    def __init__(self, arr, df, transform):
        self.arr = arr
        self.labels = df[CLASSES].values.astype(np.float32)
        self.masks = df[MASKS].values.astype(np.float32)
        self.transform = transform

    def __len__(self):
        return len(self.arr)

    def __getitem__(self, idx):
        img = self.transform(image=self.arr[idx])['image']
        return img, torch.from_numpy(self.labels[idx]), torch.from_numpy(self.masks[idx])

def worker_init_fn(_):
    """Deterministic but different seed per worker AND per epoch."""
    s = torch.initial_seed() % (2 ** 32)
    random.seed(s)
    np.random.seed(s)
    tf = torch.utils.data.get_worker_info().dataset.transform
    if hasattr(tf, 'set_random_seed'):
        try:
            tf.set_random_seed(s)
        except Exception:
            pass

BATCH_SIZE = 16   # use 8 if you get out-of-memory

def make_train_loader(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(CachedDataset(train_arr, train_df, train_tfms), batch_size=BATCH_SIZE,
                      shuffle=True, generator=g, worker_init_fn=worker_init_fn,
                      num_workers=2, pin_memory=True, drop_last=True)

def make_eval_loader(arr, df):
    return DataLoader(CachedDataset(arr, df, eval_tfms), batch_size=BATCH_SIZE,
                      shuffle=False, num_workers=2, pin_memory=True)

val_loader  = make_eval_loader(val_arr, val_df)
test_loader = make_eval_loader(test_arr, test_df)
ext_loader  = make_eval_loader(ext_arr, ext_df)

imgs, labels, masks = next(iter(make_train_loader(42)))
print("Batch:", imgs.shape, labels.shape, masks.shape)

fig, ax = plt.subplots(1, 4, figsize=(18, 5))
for j, (d, name) in enumerate([(odir_train, "ODIR"), (rfmid["train"], "RFMiD")]):
    s = d.iloc[0]
    k = train_df.index[train_df['path'] == s['path']][0]
    ax[2*j].imshow(cv2.cvtColor(cv2.imread(s['path']), cv2.COLOR_BGR2RGB)); ax[2*j].set_title(f"{name} original")
    ax[2*j+1].imshow(train_arr[k]); ax[2*j+1].set_title(f"{name} crop + CLAHE")
for a in ax: a.axis('off')
plt.show()

In [ ]:
import torch.nn.functional as F

USE_AMP = True    # set False if NaN appears (slower but fully stable)
use_amp = USE_AMP and device.type == 'cuda'

class MaskedBCE(nn.Module):
    """BCE with pos_weight that ignores unknown labels (mask == 0)."""
    def __init__(self, pos_weight):
        super().__init__()
        self.register_buffer('pos_weight', pos_weight)

    def forward(self, logits, targets, mask):
        loss = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pos_weight, reduction='none')
        return (loss * mask).sum() / mask.sum().clamp(min=1.0)

def state_finite(m):
    """True if all weights and BatchNorm buffers are finite (no NaN/inf)."""
    return all(torch.isfinite(t).all().item()
               for t in list(m.parameters()) + list(m.buffers()) if t.is_floating_point())

def train_one_epoch(model, loader, criterion, optimizer, scaler, ema=None, clip=2.0):
    model.train()
    running_loss, n, skipped = 0.0, 0, 0
    for imgs, labels, masks in tqdm(loader, leave=False, desc="train"):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            logits = model(imgs)
        loss = criterion(logits.float(), labels, masks)
        if not torch.isfinite(loss):
            skipped += 1
            continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        scaler.step(optimizer)
        scaler.update()
        if ema is not None:
            ema.update_parameters(model)
        running_loss += loss.item() * imgs.size(0)
        n += imgs.size(0)
    if skipped:
        print(f"  (skipped {skipped} non-finite batches)")
    return running_loss / max(n, 1)

@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluation in fp32 (safer than fp16)."""
    model.eval()
    running_loss, n = 0.0, 0
    probs_all, labels_all, masks_all = [], [], []
    for imgs, labels, masks in tqdm(loader, leave=False, desc="eval"):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        logits = model(imgs).float()
        running_loss += criterion(logits, labels, masks).item() * imgs.size(0)
        n += imgs.size(0)
        probs_all.append(torch.sigmoid(logits).cpu().numpy())
        labels_all.append(labels.cpu().numpy())
        masks_all.append(masks.cpu().numpy())
    return running_loss / n, np.concatenate(probs_all), np.concatenate(labels_all), np.concatenate(masks_all)

@torch.no_grad()
def predict_tta(model, x):
    """Average sigmoid probabilities over 4 views: original, h-flip, v-flip, both flips."""
    views = [x, torch.flip(x, dims=[3]), torch.flip(x, dims=[2]), torch.flip(x, dims=[2, 3])]
    return torch.stack([torch.sigmoid(model(v).float()) for v in views]).mean(0)

def _known(y_true, y_prob, mask, i):
    k = mask[:, i] > 0 if mask is not None else np.ones(len(y_true), bool)
    return y_true[k, i], y_prob[k, i]

def per_class_auc(y_true, y_prob, mask=None):
    out = {}
    for i, c in enumerate(CLASSES):
        t, p = _known(y_true, y_prob, mask, i)
        ok = 0 < t.sum() < len(t)
        out[c] = roc_auc_score(t, p) if ok else float('nan')
    return out

def find_best_thresholds(y_true, y_prob, mask=None):
    th = {}
    for i, c in enumerate(CLASSES):
        t, p = _known(y_true, y_prob, mask, i)
        best_t, best_f1 = 0.5, -1.0
        for thr in np.arange(0.05, 0.951, 0.025):
            f1 = f1_score(t, (p >= thr).astype(int), zero_division=0)
            if f1 > best_f1:
                best_t, best_f1 = thr, f1
        th[c] = round(float(best_t), 3)
    return th

def make_pos_weight():
    lab = train_df[CLASSES].values
    known = train_df[MASKS].values
    pos = (lab * known).sum(0)
    neg = known.sum(0) - pos
    return torch.tensor(np.sqrt(neg / np.maximum(pos, 1)), dtype=torch.float32).to(device)

def fit(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler,
        epochs, patience, save_path, ema):
    """Early stopping on val macro AUC (evaluating the EMA weights). Recovers from NaN automatically."""
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    best_auc, wait = 0.0, 0

    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, ema)

        if not (state_finite(model) and state_finite(ema.module)):
            print(f"Epoch {epoch:02d}: NaN in weights -> reloading best checkpoint")
            if os.path.exists(save_path):
                sd = torch.load(save_path, map_location=device)
                model.load_state_dict(sd)
                ema.module.load_state_dict(sd)
            scheduler.step()
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break
            continue

        va_loss, vp, vl, vm = evaluate(ema, val_loader, criterion)
        scheduler.step()
        macro_auc = np.nanmean(list(per_class_auc(vl, vp, vm).values()))
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['val_auc'].append(macro_auc)
        print(f"Epoch {epoch:02d}/{epochs} | train {tr_loss:.4f} | val {va_loss:.4f} | val_macro_AUC {macro_auc:.4f}")

        if macro_auc > best_auc:
            best_auc, wait = macro_auc, 0
            torch.save(ema.module.state_dict(), save_path)
            print(f"  -> saved best (AUC {best_auc:.4f})")
        else:
            wait += 1
            if wait >= patience:
                print("Early stopping.")
                break

    if history['val_auc']:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
        ax[0].set_title('Loss'); ax[0].legend()
        ax[1].plot(history['val_auc'], color='green'); ax[1].set_title('Validation macro AUC')
        plt.show()
    return history, best_auc

print("Utilities ready. AMP:", use_amp)

In [ ]:
OUT_DIR = "/kaggle/working"
SEEDS = [42]                 # start with 1 model; add 123, 2024 later for an ensemble
EPOCHS = 20
RETRAIN = False              # True = ignore saved models in /kaggle/working and train again

def find_file(name):
    """Look for a saved file in /kaggle/working first, then in any attached input."""
    p = os.path.join(OUT_DIR, name)
    if os.path.exists(p):
        return p
    for r, _, files in os.walk(INPUT_ROOT):
        if name in files:
            return os.path.join(r, name)
    return None

def build_model(pretrained):
    return timm.create_model('efficientnet_b3', pretrained=pretrained, num_classes=NUM_CLASSES,
                             drop_rate=0.4, drop_path_rate=0.3).to(device)

def train_one_model(seed):
    name = f"b3_seed{seed}.pth"
    # only skip if THIS notebook already trained it (ignore old ver-4 weights in attached inputs)
    if not RETRAIN and os.path.exists(os.path.join(OUT_DIR, name)):
        print(f"[seed {seed}] found {name} in {OUT_DIR} -> skipping training")
        return
    print(f"\n===== Training model with seed {seed} =====")
    seed_everything(seed)
    model = build_model(pretrained=True)
    ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.997), use_buffers=True)
    criterion = MaskedBCE(make_pos_weight()).to(device)

    head_params = list(model.get_classifier().parameters())
    head_ids = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': 1e-4},
        {'params': head_params,     'lr': 1e-3},
    ], weight_decay=1e-2)
    warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=1)
    cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - 1)
    scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, cosine], milestones=[1])
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    _, best = fit(model, make_train_loader(seed), val_loader, criterion, optimizer, scheduler,
                  scaler, epochs=EPOCHS, patience=6,
                  save_path=os.path.join(OUT_DIR, name), ema=ema)
    print(f"[seed {seed}] best val macro AUC: {best:.4f}")
    del model, ema, optimizer
    torch.cuda.empty_cache()

for sd in SEEDS:
    train_one_model(sd)

In [ ]:
CONFIG_NAME = "ensemble_config.json"
FORCE_RECALC_THRESHOLDS = False   # True = recompute thresholds from validation

def load_models():
    models = []
    for sd in SEEDS:
        path = os.path.join(OUT_DIR, f"b3_seed{sd}.pth")
        assert os.path.exists(path), f"Missing checkpoint for seed {sd} in {OUT_DIR} - run the training cell"
        m = build_model(pretrained=False)
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        models.append(m)
    return models

models = load_models()
print("Loaded", len(models), "models")

@torch.no_grad()
def ens_predict(x):
    """Average TTA probabilities across all ensemble members."""
    return torch.stack([predict_tta(m, x) for m in models]).mean(0)

@torch.no_grad()
def evaluate_ens(loader):
    probs_all, labels_all, masks_all = [], [], []
    for imgs, labels, masks in tqdm(loader, leave=False, desc="ensemble eval"):
        probs_all.append(ens_predict(imgs.to(device, non_blocking=True)).cpu().numpy())
        labels_all.append(labels.numpy())
        masks_all.append(masks.numpy())
    return np.concatenate(probs_all), np.concatenate(labels_all), np.concatenate(masks_all)

val_probs, val_labels, val_masks = evaluate_ens(val_loader)

# Thresholds: only reuse a config written by THIS notebook (old ver-4 thresholds don't fit the new model)
cfg_path = os.path.join(OUT_DIR, CONFIG_NAME)
if os.path.exists(cfg_path) and not FORCE_RECALC_THRESHOLDS:
    THRESHOLDS = json.load(open(cfg_path))['thresholds']
    print("Loaded saved thresholds from", cfg_path)
else:
    THRESHOLDS = find_best_thresholds(val_labels, val_probs, val_masks)
    json.dump({'thresholds': THRESHOLDS, 'seeds': SEEDS}, open(cfg_path, 'w'), indent=2)
    print("Computed and saved thresholds ->", cfg_path)
print("Thresholds:", THRESHOLDS)

def report(name, loader):
    probs, labels, masks = evaluate_ens(loader)
    aucs = per_class_auc(labels, probs, masks)
    rows = []
    for i, c in enumerate(CLASSES):
        t, p = _known(labels, probs, masks, i)
        pred = (p >= THRESHOLDS[c]).astype(int)
        tp = int(((pred == 1) & (t == 1)).sum()); fp = int(((pred == 1) & (t == 0)).sum())
        fn = int(((pred == 0) & (t == 1)).sum())
        prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-9)
        rows.append({'class': c, 'name': CLASS_NAMES[c], 'AUC': aucs[c], 'precision': prec,
                     'recall': rec, 'F1': f1, 'support': int(t.sum()), 'known': len(t)})
    tab = pd.DataFrame(rows).set_index('class')
    has_pos = tab['support'] > 0
    print(f"\n========== {name} ==========")
    print(f"Macro AUC : {np.nanmean(tab['AUC']):.4f}")
    print(f"Macro F1  : {tab.loc[has_pos, 'F1'].mean():.4f}  (classes with positives)")
    if masks.min() == 1:
        preds = np.stack([(probs[:, i] >= THRESHOLDS[c]).astype(int) for i, c in enumerate(CLASSES)], 1)
        print(f"Exact-match accuracy: {(preds == labels).all(axis=1).mean():.4f}")
    display(tab.round(3))
    return tab

odir_report  = report("ODIR internal test (compare with ver-4)", test_loader)
rfmid_report = report("RFMiD external test (never trained on)", ext_loader)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

cam = GradCAM(model=models[0], target_layers=[models[0].blocks[-1]])

def get_cam(rgb, x, class_idx):
    g = cam(input_tensor=x, targets=[ClassifierOutputTarget(class_idx)])[0]
    return show_cam_on_image(rgb.astype(np.float32) / 255.0, g, use_rgb=True)

def predict_probs(rgb):
    x = to_tensor(rgb).to(device)
    p = ens_predict(x)[0].cpu().numpy()
    return p, x

def show_examples(df, per_class=1, seed=0, cols=4):
    """Test examples per disease: filename, true labels, predictions, heatmap."""
    parts = []
    for c in CLASSES:
        sub = df[df[c] == 1]
        if len(sub):
            parts.append(sub.sample(min(per_class, len(sub)), random_state=seed))
    samples = pd.concat(parts).drop_duplicates('filename').reset_index(drop=True)

    records = []
    for start in range(0, len(samples), cols):
        chunk = samples.iloc[start:start + cols]
        fig, axes = plt.subplots(2, cols, figsize=(4.5 * cols, 9))
        axes = np.atleast_2d(axes)
        for j in range(cols):
            axes[0, j].axis('off'); axes[1, j].axis('off')
        for j, (_, s) in enumerate(chunk.iterrows()):
            rgb = load_rgb(s['path'])
            probs, x = predict_probs(rgb)
            true = [c for c in CLASSES if s[c] == 1]
            pred = [c for i, c in enumerate(CLASSES) if probs[i] >= THRESHOLDS[c]]
            top = int(np.argmax(probs))
            pred_txt = ", ".join(f"{c} {probs[CLASSES.index(c)]*100:.0f}%" for c in pred) or \
                       f"none (top: {CLASSES[top]} {probs[top]*100:.0f}%)"
            if set(pred) == set(true):  color, verdict = "green", "CORRECT"
            elif set(pred) & set(true): color, verdict = "darkorange", "PARTIAL"
            else:                       color, verdict = "red", "WRONG"

            axes[0, j].imshow(rgb)
            axes[0, j].set_title(f"{s['filename']}\nTrue: {', '.join(true)}", fontsize=11)
            axes[1, j].imshow(get_cam(rgb, x, top))
            axes[1, j].set_title(f"{verdict}\nPred: {pred_txt}", fontsize=11, color=color)
            records.append({'filename': s['filename'], 'true': true, 'pred': pred, 'verdict': verdict})
        plt.tight_layout()
        plt.show()
    return pd.DataFrame(records)

examples = show_examples(test_df, per_class=1, seed=0)
examples

In [ ]:
def analyze_rgb(rgb_in, crop=True, top_k=3, min_prob=0.15):
    """Stage 1 analysis of any RGB numpy image (can come from outside the dataset)."""
    rgb = prepare_rgb(rgb_in, crop=crop)
    probs, x = predict_probs(rgb)

    detected = [c for i, c in enumerate(CLASSES) if probs[i] >= THRESHOLDS[c]]
    if detected:
        status, shown = "DETECTED", detected
    else:
        status = "SUSPECTED (below threshold)"
        order = np.argsort(-probs)[:top_k]
        shown = [CLASSES[i] for i in order if probs[i] >= min_prob]

    top_idx = int(np.argmax(probs))
    return {'probs': probs, 'detected': detected, 'shown': shown, 'status': status,
            'heatmap': get_cam(rgb, x, top_idx), 'top': CLASSES[top_idx],
            'trigger_stage2': 'D' in detected}

# Consistency check: the same image must give exactly the same numbers every time
_img = cv2.cvtColor(cv2.imread(test_df.iloc[0]['path']), cv2.COLOR_BGR2RGB)
r1, r2 = analyze_rgb(_img), analyze_rgb(_img)
print("Same input -> same output:", np.allclose(r1['probs'], r2['probs']))

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr

def gradio_predict(img):
    if img is None:
        return None, {}, "Please upload a fundus image."
    r = analyze_rgb(img, crop=True)
    label_dict = {f"{CLASS_NAMES[c]} ({c})": float(r['probs'][i]) for i, c in enumerate(CLASSES)}
    lines = [f"**Status:** {r['status']}"]
    for c in r['shown']:
        lines.append(f"- {CLASS_NAMES[c]}: **{r['probs'][CLASSES.index(c)]*100:.1f}%**")
    if not r['shown']:
        lines.append("- No disease above the minimum probability.")
    lines.append(f"\n**Trigger Stage 2 (Diabetic Retinopathy analysis):** {r['trigger_stage2']}")
    lines.append("\n*Research prototype only, not a medical diagnosis.*")
    return r['heatmap'], label_dict, "\n".join(lines)

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Image(type="numpy", label="Upload a fundus image"),
    outputs=[gr.Image(label="Grad-CAM (where the model looked)"),
             gr.Label(num_top_classes=8, label="Probabilities"),
             gr.Markdown(label="Report")],
    title="Eye Disease Classifier - Stage 1 (EfficientNet-B3 Ensemble)",
    description="Upload a retinal fundus photo to get disease probabilities and a Grad-CAM heatmap.",
)
demo.launch(share=True)